# BWF Player Lookup

Type a badminton player's name and get their **personal details** (nationality, height, playing hand) and **ranking** (current rank and how long they have held it) from bwfbadminton.com. Section 4 downloads the player's **tournament history for the last year**: the result in each tournament, who they played with, who they played against, and the score of every game. It is saved to a database and CSV files.

**How to use:** set `PLAYER_NAME` in section 1, then *Run All*. Every value the site does not list is shown as `null`, with a note. Section 4 makes 20-40 requests for a busy player (about a minute or two at the polite pace; repeating it is quick because responses are cached).

The notebook is a thin interface: all logic lives in the `bwf_player` package (see the README).

In [1]:
import logging
import sqlite3

from bwf_player import (
    BlockedByCloudflareError,
    BwfClientError,
    download_player_history,
    format_history,
    format_result,
    lookup_player,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Enter a player name

In [2]:
# Any reasonable spelling works: case, accents, reversed order, small typos ("jonathan cristie").
PLAYER_NAME = "jonathan cristie"

# None = the first ranking event the site lists (usually singles). To pick another event, copy the
# id shown as "Other event" in the result, e.g. "9-90070" (a doubles event).
EVENT_ID = None

## 2. Result

In [3]:
try:
    result = lookup_player(PLAYER_NAME, event_id=EVENT_ID)
except BlockedByCloudflareError as exc:
    result = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    result = None
    print(f"The request failed: {exc}")
else:
    print(format_result(result))

Search:         FOUND - Matched 'Jonatan CHRISTIE' (score 94).
Profile URL:    https://bwfbadminton.com/player/73442/jonatan-christie

Personal details
  Name:          Jonatan CHRISTIE
  Nationality:   Indonesia
  Height:        179.0 cm
  Playing hand:  Right

Ranking (MEN'S SINGLES)
  Current rank:  1
  At this rank:  4 week(s), since 2026-08-25 (latest ranking list 2026-09-15)


## 3. The same result as structured data (JSON)

In [4]:
print(result.model_dump_json(indent=2) if result else "No result.")

{
  "search": {
    "query": "jonathan cristie",
    "status": "found",
    "best_match": {
      "player_id": "73442",
      "slug": "jonatan-christie",
      "name": "Jonatan CHRISTIE",
      "country": null,
      "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
      "score": 93.8
    },
    "candidates": [
      {
        "player_id": "73442",
        "slug": "jonatan-christie",
        "name": "Jonatan CHRISTIE",
        "country": null,
        "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
        "score": 93.8
      }
    ],
    "message": "Matched 'Jonatan CHRISTIE' (score 94)."
  },
  "profile": {
    "player_id": "73442",
    "player_found": true,
    "name": "Jonatan CHRISTIE",
    "nationality": "Indonesia",
    "height_cm": 179.0,
    "playing_hand": "Right",
    "missing_fields": [],
    "notes": []
  },
  "ranking": {
    "player_id": "73442",
    "event": {
      "id": "6-0",
      "name": "MEN'S SINGLES"
    },
    "o

## 4. Tournament history for the last year

Downloads every tournament the player entered in the window and, for each event, all matches: the **result** (`1st`, `QF`, `R16`, ...), the **partner** (doubles), the **opponents** and the **points of every game**. The data is saved to a SQLite database and exported as CSV.

- `HISTORY_PLAYER`: a name, or the site's player id as a number.
- `SINCE` / `UNTIL`: `None` means the last 12 months up to today. A tournament counts if its dates overlap the window.
- Running it again never duplicates anything; it updates the same rows.

In [5]:
HISTORY_PLAYER = PLAYER_NAME      # or the site's player id, e.g. 73442

# None = one year back from today, up to today. Or pass datetime.date values.
SINCE = UNTIL = None

# Where to save (the data/ folder is git-ignored).
DB_PATH = "data/bwf_history.sqlite"
CSV_DIR = "data/export"

In [6]:
try:
    history = download_player_history(
        HISTORY_PLAYER, since=SINCE, until=UNTIL, db_path=DB_PATH, export_dir=CSV_DIR
    )
except BlockedByCloudflareError as exc:
    history = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    history = None
    print(f"The download failed: {exc}")
else:
    print(format_history(history))

INFO bwf_player.history: Looking up the tournaments of player 73442


INFO bwf_player.http_client: Bootstrapping session cookie


INFO bwf_player.history: [1/19] 2025-09-16 LI-NING China Masters 2025 (MS)


INFO bwf_player.history: [2/19] 2025-09-23 SUWON VICTOR Korea Open 2025 (MS)


INFO bwf_player.history: [3/19] 2025-10-14 VICTOR Denmark Open 2025 (MS)


...

INFO bwf_player.history: [16/19] 2026-07-14 DAIHATSU Japan Open 2026 (MS)


INFO bwf_player.history: [17/19] 2026-07-21 VICTOR China Open 2026 (MS)


INFO bwf_player.history: [18/19] 2026-08-17 BWF World Championships 2026 (MS)


INFO bwf_player.history: [19/19] 2026-09-01 LI-NING China Masters 2026 (MS)


Search:       FOUND - Matched 'Jonatan CHRISTIE' (score 94).
Player:       Jonatan CHRISTIE (id 73442)
Window:       2025-09-21 to 2026-09-21
Downloaded:   19 tournament(s), 19 event(s), 58 match(es) (58 played), 138 game(s)
Checked:      the matches reproduce the site's own totals: yes (19 event(s))
Database:     data\bwf_history.sqlite
CSV:          data\export\results.csv
CSV:          data\export\matches.csv
CSV:          data\export\games.csv

2025-09-16  LI-NING China Masters 2025  [MS]  result: R16  1-1 in matches  (HSBC BWF World Tour Super 750)
    R32       won              vs Kenta NISHIMOTO  21-19, 21-19
    R16       lost             vs LIN Chun-Yi  5-21, 20-22

2025-09-23  SUWON VICTOR Korea Open 2025  [MS]  result: 1st  5-0 in matches  (HSBC BWF World Tour Super 500)
    R32       won              vs NG Ka Long Angus  21-11, 21-17
    R16       won              vs Chia Hao LEE  22-20, 15-21, 21-15
    QF        won              vs Kenta NISHIMOTO  21-14, 21-8
    SF     

### Read the saved data back

The database has the tables `players`, `tournaments`, `results`, `matches`, `match_players` and `games`, and the view `player_match_view` with one row per player and match (partner, opponents, games, won). For analysis, load the CSV files or the view, for example `pandas.read_sql("SELECT * FROM player_match_view", sqlite3.connect(DB_PATH))`.

In [7]:
if history and history.database:
    with sqlite3.connect(history.database) as connection:
        rows = connection.execute(
            """SELECT match_date, tournament, event_code, round, won, partner, opponent_1, opponent_2, games
               FROM player_match_view WHERE player_id = ? ORDER BY match_date, seq LIMIT 12""",
            (int(history.player_id),),
        ).fetchall()
    print("First matches in the database (date, tournament, event, round, won, partner, opponents, games):")
    for row in rows:
        print("  ", " | ".join("-" if value is None else str(value) for value in row))
    print("\nCSV files:", *history.csv_files.values(), sep="\n  ")
else:
    print("Nothing was saved (see the messages above).")

First matches in the database (date, tournament, event, round, won, partner, opponents, games):
   2025-09-17 | LI-NING China Masters 2025 | MS | R32 | 1 | - | Kenta NISHIMOTO | - | 21-19, 21-19
   2025-09-18 | LI-NING China Masters 2025 | MS | R16 | 0 | - | LIN Chun-Yi | - | 5-21, 20-22
   2025-09-24 | SUWON VICTOR Korea Open 2025 | MS | R32 | 1 | - | NG Ka Long Angus | - | 21-11, 21-17
   2025-09-25 | SUWON VICTOR Korea Open 2025 | MS | R16 | 1 | - | Chia Hao LEE | - | 22-20, 15-21, 21-15
   2025-09-26 | SUWON VICTOR Korea Open 2025 | MS | QF | 1 | - | Kenta NISHIMOTO | - | 21-14, 21-8
   2025-09-27 | SUWON VICTOR Korea Open 2025 | MS | SF | 1 | - | Alwi FARHAN | - | 18-21, 21-14, 21-15
   2025-09-28 | SUWON VICTOR Korea Open 2025 | MS | Final | 1 | - | Anders ANTONSEN | - | 21-10, 15-21, 21-17
   2025-10-15 | VICTOR Denmark Open 2025 | MS | R32 | 1 | - | Kenta NISHIMOTO | - | 10-21, 21-11, 21-7
   2025-10-16 | VICTOR Denmark Open 2025 | MS | R16 | 1 | - | Kodai NARAOKA | - | 21-7, 2

## More examples

The same lookup for other cases: a left-handed player, a retired (unranked) player, a player whose profile lists nothing, a name that matches several players, and a name that matches nobody.

In [8]:
EXAMPLES = ["Carolina Marin", "Chong Wei Lee", "Aadhya Shine", "christie", "not a real player"]

for name in EXAMPLES:
    print("=" * 72)
    print(f"Query: {name!r}")
    try:
        print(format_result(lookup_player(name)))
    except BwfClientError as exc:
        print(f"The request failed: {exc}")

Query: 'Carolina Marin'
Search:         FOUND - Matched 'Carolina MARIN' (score 100).
Profile URL:    https://bwfbadminton.com/player/18228/carolina-marin

Personal details
  Name:          Carolina MARIN
  Nationality:   Spain
  Height:        172.0 cm
  Playing hand:  Left

Ranking (WOMEN'S SINGLES)
  Current rank:  null
  At this rank:  null
  Other event:   WOMEN'S DOUBLES (Beatriz CORRALES) [9-95780]
  Other event:   WOMEN'S DOUBLES (Sara PEÑALVER) [9-76747]
  Other event:   WOMEN'S DOUBLES (Isabel FERNANDEZ) [9-90558]
  Other event:   WOMEN'S DOUBLES (Clara AZURMENDI) [9-74218]
  Other event:   WOMEN'S DOUBLES (Ana Maria MARTIN) [9-55582]

Notes
  - Not currently ranked in WOMEN'S SINGLES: the site lists no current rank.
Query: 'Chong Wei Lee'


Search:         FOUND - Matched 'LEE Chong Wei' (score 100).
Profile URL:    https://bwfbadminton.com/player/50152/lee-chong-wei

Personal details
  Name:          LEE Chong Wei
  Nationality:   Malaysia
  Height:        172.0 cm
  Playing hand:  Right

Ranking (MEN'S SINGLES)
  Current rank:  null
  At this rank:  null

Notes
  - Not currently ranked in MEN'S SINGLES: the site lists no current rank.
Query: 'Aadhya Shine'
Search:         FOUND - Matched 'Aadhya SHINE' (score 100).
Profile URL:    https://bwfbadminton.com/player/89438/aadhya-shine

Personal details
  Name:          Aadhya SHINE
  Nationality:   null
  Height:        null
  Playing hand:  null

Ranking (WOMEN'S SINGLES)
  Current rank:  423
  At this rank:  1 week(s), since 2026-09-15 (latest ranking list 2026-09-15)
  Other event:   WOMEN'S DOUBLES (Nanda GHOSH) [9-90070]

Notes
  - nationality is not listed on the player's profile.
  - height is not listed on the player's profile.
  - playing hand is not listed on the 

Search:         AMBIGUOUS - Several players match closely; choose one of the candidates.
   90.0  Christie XU (Canada)  https://bwfbadminton.com/player/51260/christie-xu
   90.0  Jonatan CHRISTIE (Indonesia)  https://bwfbadminton.com/player/73442/jonatan-christie
   90.0  Trehan CHRISTIE (England)  https://bwfbadminton.com/player/75315/trehan-christie
Several players match; re-run with a fuller name to pick one.
Query: 'not a real player'
Search:         NOT_FOUND - No player matched 'not a real player' at or above 85. Closest was 'Emil DANTLER' (62).


## Notes

- **Search** ignores case, accents and word order, and tolerates typos in full names. A partial name such as "christie" returns ranked candidates instead of guessing.
- **Weeks at this rank** is derived from the site's weekly ranking history: the number of consecutive weekly lists, ending with the latest, that show the current rank. (The site's own "consecutive weeks" figure describes the player's *best* rank, so it is not used.)
- **null** means the site lists no value. A player with no current rank (retired, or inactive for a long time) is reported as unranked.
- Responses are cached under `.cache/bwf_player` (ranking and profile for 24 h, the player index for 7 days), so re-running a cell makes no new requests. The site sits behind Cloudflare bot protection; the client rate-limits itself and stops at once if it is blocked.
- Known limitations and the full design are in `docs/PRD_master.md`.
- **Tournament history** (section 4): `won` is `1` or `0`, or empty for a bye (the player advanced without playing; the site counts it as a match won). A partner is empty in singles. The check line says whether the downloaded matches add up to the totals the site itself shows for each tournament.
